In [1]:

import os, json, argparse, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             roc_auc_score, roc_curve, auc, confusion_matrix,
                             classification_report)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)



In [2]:


#  1. CONFIG
train_file = "CICIOT23/train/train.csv"
val_file   = "CICIOT23/validation/validation.csv"
test_file  = "CICIOT23/test/test.csv"
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)




In [3]:


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--task", choices=["binary", "multiclass"], default="binary")
    p.add_argument("--sample", type=float, default=1.0, help="fraction of rows to load per split")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch", type=int, default=1024)
    p.add_argument("--baseline", action="store_true", help="also train a Random Forest baseline")
    p.add_argument("--skip_eda", action="store_true")
    return p.parse_args()




In [4]:
#  2. DATA LOADING
def load_split(path, frac):
    df = pd.read_csv(path)
    if frac < 1.0:
        df = df.sample(frac=frac, random_state=SEED)
    return df.reset_index(drop=True)


def find_label_col(df):
    for c in df.columns:
        if c.strip().lower() == "label":
            return c
    raise ValueError("Label column not found: " + str(list(df.columns)[-5:]))


def to_category(label):
    """Map the 34 fine-grained labels to 8 families (Benign + 7 attack types)."""
    l = str(label)
    if l.startswith("Benign"):                       return "Benign"
    if l.startswith("DDoS"):                         return "DDoS"
    if l.startswith("DoS"):                          return "DoS"
    if l.startswith("Mirai"):                        return "Mirai"
    if l.startswith("Recon") or l == "VulnerabilityScan": return "Recon"
    if l.startswith("MITM") or l == "DNS_Spoofing":  return "Spoofing"
    if l == "DictionaryBruteForce":                  return "BruteForce"
    return "Web-based"  # XSS, SqlInjection, CommandInjection, BrowserHijacking, Backdoor_Malware, Uploading_Attack




In [5]:
# - 3. EDA
def run_eda(train_df, label_col):
    print("\n[EDA] shape:", train_df.shape)
    print("[EDA] missing values:", int(train_df.isna().sum().sum()))
    print("[EDA] duplicate rows:", int(train_df.duplicated().sum()))
    print(train_df.describe().T.head(15))

    # (a) fine-grained class distribution
    plt.figure(figsize=(9, 9))
    train_df[label_col].value_counts().sort_values().plot.barh(color="steelblue")
    plt.xscale("log"); plt.xlabel("samples (log scale)"); plt.title("Class distribution (34 labels)")
    plt.tight_layout();  plt.close()

    # (b) family distribution
    fam = train_df[label_col].map(to_category)
    plt.figure(figsize=(7, 4))
    fam.value_counts().plot.bar(color="darkorange")
    plt.yscale("log"); plt.title("Attack family distribution"); plt.ylabel("samples (log)")
    plt.tight_layout();  plt.close()

    # (c) benign vs attack ratio
    ratio = (train_df[label_col] != "BenignTraffic").value_counts(normalize=True)
    print("[EDA] attack ratio:\n", ratio)

    # (d) correlation heat-map on a sample
    num = train_df.select_dtypes(include=np.number).sample(min(50000, len(train_df)), random_state=SEED)
    plt.figure(figsize=(14, 11))
    sns.heatmap(num.corr(), cmap="coolwarm", center=0, xticklabels=True, yticklabels=True)
    plt.title("Feature correlation matrix"); plt.tight_layout()
    plt.close()

    # (e) histograms of the 8 features most correlated with "is attack"
    y_bin = (train_df[label_col] != "BenignTraffic").astype(int)
    corr = train_df.select_dtypes(include=np.number).corrwith(y_bin).abs().sort_values(ascending=False)
    top = corr.head(8).index.tolist()
    fig, axes = plt.subplots(2, 4, figsize=(18, 7))
    s = train_df.sample(min(50000, len(train_df)), random_state=SEED)
    sy = (s[label_col] != "BenignTraffic")
    for ax, f in zip(axes.ravel(), top):
        ax.hist(np.log1p(s.loc[~sy, f].clip(lower=0)), bins=40, alpha=.6, label="Benign")
        ax.hist(np.log1p(s.loc[sy, f].clip(lower=0)), bins=40, alpha=.6, label="Attack")
        ax.set_title(f); ax.set_yscale("log")
    axes[0, 0].legend(); plt.suptitle("Top-8 features vs target (log1p scale)")
    plt.tight_layout();  plt.close()
    print("[EDA] top correlated features:\n", corr.head(8))




In [6]:
#  4. PREPROCESSING
def preprocess(train_df, val_df, test_df, task):
    lc = find_label_col(train_df)
    # remove duplicates in TRAIN only (avoid leakage inflating the score)
    before = len(train_df)
    train_df = train_df.drop_duplicates().reset_index(drop=True)
    print(f"[PRE] dropped {before - len(train_df)} duplicate training rows")

    def split_xy(df):
        y_raw = df[lc].astype(str)
        X = df.drop(columns=[lc]).select_dtypes(include=np.number).replace([np.inf, -np.inf], np.nan)
        return X, y_raw

    Xtr, ytr_raw = split_xy(train_df)
    Xva, yva_raw = split_xy(val_df)
    Xte, yte_raw = split_xy(test_df)

    # drop constant columns, fill NaN with train median
    keep = [c for c in Xtr.columns if Xtr[c].nunique() > 1]
    Xtr, Xva, Xte = Xtr[keep], Xva[keep], Xte[keep]
    med = Xtr.median()
    Xtr, Xva, Xte = Xtr.fillna(med), Xva.fillna(med), Xte.fillna(med)

    # scale: fit on train only
    scaler = StandardScaler().fit(Xtr)
    Xtr, Xva, Xte = [np.clip(scaler.transform(x), -10, 10).astype("float32") for x in (Xtr, Xva, Xte)]

    # targets
    if task == "binary":
        f = lambda s: (s != "BenignTraffic").astype(int).values
        ytr, yva, yte = f(ytr_raw), f(yva_raw), f(yte_raw)
        class_names = ["Benign", "Attack"]
    else:
        le = LabelEncoder().fit(ytr_raw.map(to_category))
        f = lambda s: le.transform(s.map(to_category))
        ytr, yva, yte = f(ytr_raw), f(yva_raw), f(yte_raw)
        class_names = list(le.classes_)
    print(f"[PRE] features: {len(keep)} | train {Xtr.shape} val {Xva.shape} test {Xte.shape}")
    print("[PRE] train class counts:", dict(zip(*np.unique(ytr, return_counts=True))))
    return Xtr, Xva, Xte, ytr, yva, yte, class_names, keep




In [7]:
#  5. MODELS
def build_mlp(n_feat, n_cls):
    out = layers.Dense(1, activation="sigmoid") if n_cls == 2 else layers.Dense(n_cls, activation="softmax")
    m = models.Sequential([
        layers.Input(shape=(n_feat,)),
        layers.Dense(128), layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.3),
        layers.Dense(64),  layers.BatchNormalization(), layers.Activation("relu"), layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        out], name="MLP")
    return m


def build_cnn(n_feat, n_cls):
    out = layers.Dense(1, activation="sigmoid") if n_cls == 2 else layers.Dense(n_cls, activation="softmax")
    m = models.Sequential([
        layers.Input(shape=(n_feat, 1)),
        layers.Conv1D(64, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, padding="same", activation="relu"), layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, 3, padding="same", activation="relu"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation="relu"), layers.Dropout(0.3),
        out], name="CNN1D")
    return m


def train_model(model, Xtr, ytr, Xva, yva, n_cls, epochs, batch):
    loss = "binary_crossentropy" if n_cls == 2 else "sparse_categorical_crossentropy"
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
    cw = compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)
    cw = dict(zip(np.unique(ytr), cw))
    cbs = [callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
           callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)]
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=epochs, batch_size=batch,
                     class_weight=cw, callbacks=cbs, verbose=2)
    return hist




In [8]:
#  6. EVALUATION
def evaluate(name, y_true, prob, class_names, task):
    n_cls = len(class_names)
    if task == "binary":
        prob = prob.ravel(); pred = (prob >= 0.5).astype(int)
        res = dict(accuracy=accuracy_score(y_true, pred),
                   precision=precision_score(y_true, pred),
                   recall=recall_score(y_true, pred),
                   f1=f1_score(y_true, pred),
                   auc=roc_auc_score(y_true, prob))
        fpr, tpr, _ = roc_curve(y_true, prob)
        plt.figure(figsize=(5.5, 5)); plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.4f})")
        plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title(f"ROC curve - {name}"); plt.legend(); plt.tight_layout()
    else:
        pred = prob.argmax(1)
        res = dict(accuracy=accuracy_score(y_true, pred),
                   precision=precision_score(y_true, pred, average="macro", zero_division=0),
                   recall=recall_score(y_true, pred, average="macro", zero_division=0),
                   f1=f1_score(y_true, pred, average="macro", zero_division=0),
                   precision_weighted=precision_score(y_true, pred, average="weighted", zero_division=0),
                   recall_weighted=recall_score(y_true, pred, average="weighted", zero_division=0),
                   f1_weighted=f1_score(y_true, pred, average="weighted", zero_division=0))
        present = [i for i in range(n_cls) if (y_true == i).any()]
        yb = label_binarize(y_true, classes=range(n_cls))
        res["auc"] = roc_auc_score(yb[:, present], prob[:, present], average="macro")
        plt.figure(figsize=(7, 6))
        for i in present:
            fpr, tpr, _ = roc_curve(yb[:, i], prob[:, i])
            plt.plot(fpr, tpr, label=f"{class_names[i]} ({auc(fpr, tpr):.3f})")
        plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("FPR"); plt.ylabel("TPR")
        plt.title(f"One-vs-Rest ROC - {name} (macro AUC={res['auc']:.4f})"); plt.legend(fontsize=7)
        plt.tight_layout()
    plt.savefig(f"{OUT}/roc_{name}_{task}.png", dpi=150); plt.close()
 
    cm = confusion_matrix(y_true, pred)
    plt.figure(figsize=(6.5, 5.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion matrix - {name}")
    plt.tight_layout(); plt.savefig(f"{OUT}/cm_{name}_{task}.png", dpi=150); plt.close()
 
    rep = classification_report(y_true, pred, target_names=class_names, zero_division=0)
    print(f"\n===== {name} ({task}) test results =====")
    print(rep)
    print({k: round(float(v), 5) for k, v in res.items()})
    with open(f"{OUT}/report_{name}_{task}.txt", "w") as f:
        f.write(rep)
    return {k: float(v) for k, v in res.items()}
 
 
def plot_history(hist, name, task):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val")
    ax[0].set_title(f"{name} loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val")
    ax[1].set_title(f"{name} accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.savefig(f"{OUT}/history_{name}_{task}.png", dpi=150); plt.close()
 
 


In [9]:
#  7. MAIN
def main():
    a = parse_args()
    print("Loading data ...")
    tr, va, te = load_split(train_file, a.sample), load_split(val_file, a.sample), load_split(test_file, a.sample)
    if not a.skip_eda:
        run_eda(tr, find_label_col(tr))
 
    Xtr, Xva, Xte, ytr, yva, yte, names, feats = preprocess(tr, va, te, a.task)
    n_feat, n_cls = Xtr.shape[1], len(names)
    results = {}
 
    # ---- Model 1: MLP
    mlp = build_mlp(n_feat, n_cls); mlp.summary()
    h = train_model(mlp, Xtr, ytr, Xva, yva, n_cls, a.epochs, a.batch)
    plot_history(h, "MLP", a.task)
    results["MLP"] = evaluate("MLP", yte, mlp.predict(Xte, batch_size=4096, verbose=0), names, a.task)
 
    # ---- Model 2: 1D-CNN (features treated as a 1-D sequence with one channel)
    cnn = build_cnn(n_feat, n_cls); cnn.summary()
    h = train_model(cnn, Xtr[..., None], ytr, Xva[..., None], yva, n_cls, a.epochs, a.batch)
    plot_history(h, "CNN1D", a.task)
    results["CNN1D"] = evaluate("CNN1D", yte, cnn.predict(Xte[..., None], batch_size=4096, verbose=0), names, a.task)
 
    # ---- Optional baseline
    if a.baseline:
        from sklearn.ensemble import RandomForestClassifier
        rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, class_weight="balanced_subsample", random_state=SEED)
        rf.fit(Xtr, ytr)
        p = rf.predict_proba(Xte)
        results["RandomForest"] = evaluate("RandomForest", yte, p[:, 1] if a.task == "binary" else p, names, a.task)
 
    
 